In [12]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import chromadb
import uuid

# loading environment varaibles
load_dotenv("../.env")
load_dotenv("../.secrets")

#creating OpenAI client
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)


In [13]:
# function for obtaining embeddings
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=text, model=model).data[0].embedding

In [14]:
# Opening database file
with open("curiosity_dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()


# Splitting data by separators.
facts = text.split("<sep>")

# Cleaning up extra spaces
cleaned_facts = []

for fact in facts:
    fact = fact.replace("\n", " ")
    fact = fact.strip()

    if fact:
        cleaned_facts.append(fact)

facts = cleaned_facts


In [15]:
# creating persistent chromadb instance
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

#creating a collection
collection = chroma_client.get_or_create_collection(
    name="fun_facts"
)

In [16]:
response = client.embeddings.create(
    input=facts,
    model="text-embedding-3-small"
)

embeddings = [item.embedding for item in response.data]

ids = [f"id{i}" for i in range(len(facts))]

collection.add(
    embeddings=embeddings,
    documents=facts,
    ids=ids
)

InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database